In [2]:
"""
Copyright (c) 2026 Baechul Kim
All rights reserved.

Author: Baechul Kim <baechul@gmail.com>
Date: April 15, 2026
Description: Pre-compute the feature table with some timeframes and lag features for training 
License: MIT
"""

### Feature Engineering ###
# To add various feature columns that would help model's learning and prediction.
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.read_csv('data/sales.csv')
df_features = df.copy() # individual transactions one row per transaction

# For training and prediction, I prefer to use a derived dataframe 
# rather than rows per transactions (df_features). Hence switching to 
# the category_sales dataframe.
# Aggregate to (date, category) level for forecasting the category_sales.
category_sales = (
    df_features.groupby(['date', 'product_category'])['total_revenue']
    .sum()
    .reset_index()
    .sort_values(['product_category', 'date'])
)

# 1) Pre-Processing: Encoding str category values because most algorithms require numerical inputs.
le = LabelEncoder()
category_sales['category_encoded'] = le.fit_transform(category_sales['product_category'])

# 2) Pre-Processing: Convert date to datetime (because date-based features are added in next)
if 'date' in category_sales.columns:
  category_sales['date'] = pd.to_datetime(category_sales['date'], errors='coerce')

# 3) Temporal Features - to have model learn time-based patterns
# Example: April tends to have higher/lower revenue than January.
category_sales['year'] = category_sales['date'].dt.year
category_sales['month'] = category_sales['date'].dt.month
category_sales['week'] = category_sales['date'].dt.isocalendar().week
category_sales['day_of_week'] = category_sales['date'].dt.dayofweek
# category_sales['day_of_month'] = df_features['date'].dt.day
# category_sales['quarter'] = df_features['date'].dt.quarter

# 4) Lag Features - to have model learn what happens before
# Example: If sale was high 7 days ago, it's likely to be high today too.
category_sales['revenue_lag_1'] = category_sales.groupby('product_category')['total_revenue'].shift(1)
category_sales['revenue_lag_7'] = category_sales.groupby('product_category')['total_revenue'].shift(7)
category_sales['revenue_lag_14'] = category_sales.groupby('product_category')['total_revenue'].shift(14)
category_sales['revenue_lag_30'] = category_sales.groupby('product_category')['total_revenue'].shift(30)

# shift() fills in NaN into the shifted - no need such rows.  
category_sales = category_sales.dropna(subset=['revenue_lag_1', 'revenue_lag_7', 'revenue_lag_14', 'revenue_lag_30'])

In [3]:
### Train/Test Data Setup ###
# For demo purpose, I use only the following limited columns. I will use 80% for train
# and 20% for validation.
feature_columns = [
    'category_encoded', 'year', 'month', 'week', 'day_of_week',
    'revenue_lag_1', 'revenue_lag_7', 'revenue_lag_14', 'revenue_lag_30'
]
target = 'total_revenue'

# train(80%)/test(20%) split
category_sales = category_sales.sort_values('date')
train_size = int(len(category_sales) * 0.8)
train_df = category_sales.iloc[:train_size]
test_df = category_sales.iloc[train_size:]

X_train = train_df[feature_columns] 
y_train = train_df[target]
X_test = test_df[feature_columns]
y_test = test_df[target]

In [4]:
### Model Selection and Training ###
 
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import lightgbm as lgb
import numpy as np

### LightGBM ###
# 1) Train
# For demo purpose, I did the repeated traing and validation processes (for-loop with
# different hyperparameters) to get the optimized ones. 
# I ended up with 200 boosting trees and 0.05 learning rate.
model_lgb = lgb.LGBMRegressor(
  n_estimators=200, 
  learning_rate=0.05, 
  max_depth=4, 
  random_state=432
)
model_lgb.fit(X_train, y_train)

# 2) Prediction for tests and repeated optimization with different hyperparameters
predictions_lgb = model_lgb.predict(X_test) 

# 3) Model Evaluation and Results
# I am using only MAE and MSE for the regression prediction evaluation for demo purpose.
mae_lgb = mean_absolute_error(y_test, predictions_lgb)
mse_lgb = mean_squared_error(y_test, predictions_lgb)
rmse_lgb = np.sqrt(mse_lgb)

print(f'MAE  (LightGBM): {mae_lgb:.2f}')
print(f'RMSE (LightGBM): {rmse_lgb:.2f}')

### XGB ###
# 1) Train
model_xgb = XGBRegressor(
  n_estimators=200, 
  learning_rate=0.05, 
  max_depth=4, 
  random_state=432
)
model_xgb.fit(X_train, y_train)

# 2) Prediction 
predictions_xgb = model_xgb.predict(X_test) 

# 3) Model Evaluation
mae_xgb = mean_absolute_error(y_test, predictions_xgb)
mse_xgb = mean_squared_error(y_test, predictions_xgb)
rmse_xgb = np.sqrt(mse_xgb)

print(f'MAE  (XGB): {mae_xgb:.2f}')
print(f'RMSE (XGB): {rmse_xgb:.2f}')

# when trained with sales-small.csv (only 240 transactions)
# MAE  (LightGBM): 165.44
# RMSE (LightGBM): 217.93
# MAE  (XGB): 143.68
# RMSE (XGB): 185.86

# when trained with sales.csv (87,473 transactions)
# MAE  (LightGBM): 4931.26
# RMSE (LightGBM): 8328.51
# MAE  (XGB): 5006.14
# RMSE (XGB): 8528.70

# My Conclusion:
# In general, LightGBM is known as fast, distributed, high-performance and memory efficient 
# gradient boosting framework better than xgboot. However, in my case, with very limited
# data (only 240 rows) and optimization, xgboot showed less errors in both MAE and MSE.
# However with the larger data (87,473 rows), the winner is LightGBM
# For prod, I would use LightGBM.

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000546 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1102
[LightGBM] [Info] Number of data points in the train set: 3360, number of used features: 9
[LightGBM] [Info] Start training from score 19160.166843
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

In [4]:
### Save Models to Artifactory ### 

import joblib

# Bundle each model with the LabelEncoder so consumers always use
# the exact same category→integer mapping that was used during training.
lgb_artifact = {
  'model': model_lgb,
  'le':    le,
  'feature_columns': feature_columns,
}
xgb_artifact = {
  'model': model_xgb,
  'le':    le,
  'feature_columns': feature_columns,
}

joblib.dump(lgb_artifact, 'models/top_sales_prediction_lgb.joblib')
joblib.dump(xgb_artifact, 'models/top_sales_prediction_xgb.joblib')

# In my craft-poc app:
#  artifact = joblib.load('models/top_sales_prediction_lgb.joblib')
#  model, le, feature_columns = artifact['model'], artifact['le'], artifact['feature_columns']
#
#  df['category_encoded'] = le.transform(df['product_category'])  # NOT .fit_transform
#  predictions = model.predict(df[feature_columns])

# END of NoteBook

['models/top_sales_prediction_xgb.joblib']